# BDM600 - Workshop 06 - Youtube analytics

## Libraries

In [138]:
import pandas
import googleapiclient.discovery

## API information

In [183]:
# API information
api_service_name = "youtube"
api_version = "v3"
developerKey = 'Insert API Key here'

In [148]:
youtube = googleapiclient.discovery.build(
    api_service_name,
    api_version,
    developerKey=developerKey
)

## Test Requesst

In [149]:
test_request = youtube.channels().list(
    part = 'statistics',
    forUsername = 'CNN'
)

In [150]:
test_response = test_request.execute()
test_response

{'kind': 'youtube#channelListResponse',
 'etag': 'gykGHNt9EMJnnVGJAp_4e4NnLXk',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': 'w4-lhhYMtWtKQhJ6RqmXvZOgFak',
   'id': 'UCupvZG-5ko_eiXAupbDfxWw',
   'statistics': {'viewCount': '19975872408',
    'subscriberCount': '19200000',
    'hiddenSubscriberCount': False,
    'videoCount': '180117'}}]}

## Getting Channel ID's

List of 10 Mainstream media Youtube Channels.

In [151]:
channel_names = [
    "CBC News",
    "USA TODAY",
    "Washington Post",
    "NBC News",
    "Reuters",
    "The Wall Street Journal",
    "CNN",
    "BBC News",
    "The New York Times",
    "Fox News"
]

In [152]:
def get_channel_id(channel_name):
  request = youtube.search().list(
    part = "snippet", #there may be a more optimal way to query for channels
    q = channel_name,
    type = "channel",
    maxResults=1
  )

  response = request.execute()

  if response['items']:
    channel_id = response['items'][0]['snippet']['channelId']
    return channel_id

  else:
    return None

channel_ids = {}

#Double check that we have the correct channels
for name in channel_names:
  channel_id = get_channel_id(name)
  channel_ids[name] = channel_id
  print(f"{name}'s link: https://www.youtube.com/channel/{channel_id}")

CBC News's link: https://www.youtube.com/channel/UCuFFtHWoLl5fauMMD5Ww2jA
USA TODAY's link: https://www.youtube.com/channel/UCP6HGa63sBC7-KHtkme-p-g
Washington Post's link: https://www.youtube.com/channel/UCHd62-u_v4DvJ8TCFtpi4GA
NBC News's link: https://www.youtube.com/channel/UCeY0bbntWzzVIaj2z3QigXg
Reuters's link: https://www.youtube.com/channel/UChqUTb7kYRX8-EiaN3XFrSQ
The Wall Street Journal's link: https://www.youtube.com/channel/UCK7tptUDHh-RYDsdxO1-5QQ
CNN's link: https://www.youtube.com/channel/UCupvZG-5ko_eiXAupbDfxWw
BBC News's link: https://www.youtube.com/channel/UC16niRr50-MSBwiO3YDb3RA
The New York Times's link: https://www.youtube.com/channel/UCqnbDFdCpuN8CMEg0VuEBqA
Fox News's link: https://www.youtube.com/channel/UCXIJgqnII2ZOINSWNOGFThA


We can manually double check that We found the exact youtube channel we want by observing their links. (not a scalable solution, but works well in this case).

In [154]:
print(len(channel_ids))

10


## PART 2 - Retrieving Metadata & Content Details:

Retrieving the 5 latest videos related to "Trump". For each of the afformentioned channels in channel_names as well as the metadata for said videos...

* Date of publication
* Channel Title
* Title
* Description
* Tags
* Caption availability
* Duration
* Default language

In [155]:
def get_videos(chan_id):
  video_query = youtube.search().list(
      part = "id",
      channelId = chan_id,
      q = "Trump",
      type = "video",
      order = "date",
      maxResults = 5
  )

  response = video_query.execute()

  video_ids = []
  for video in response['items']:
    video_ids.append(video['id']['videoId'])

  return video_ids

In [156]:
test = get_videos(channel_ids["CNN"])

In [158]:
test[5]

'JcHHIuzV-g4'

In [159]:
print(f"https://www.youtube.com/watch?v={test[0]}")

https://www.youtube.com/watch?v=SxuCj51D3ns


Query works, now to apply it to all 10 of our channel ids we extracted...

In [161]:
recent_trump = {}

#may remove the dictionary and retain a list as each video-
#id will carry the channel name when queried anyways.
for channel in channel_ids:
  recent_trump[channel] = get_videos(channel_ids[channel])

In [162]:
print(recent_trump["CNN"])

['SxuCj51D3ns', 'esaWZGjluLg', 'gseDdTQbqC8', 'Ppqc4pHG_ig', 'UPBd6pUzgHs']


Now that we have all 10 channel's top 5 videos about Trump (based on title of video, description,etc...) we can begin generating a dataset.


In [164]:
def get_video_meta_data(video_id):
  video_query = youtube.videos().list(
      part='snippet,contentDetails',
      id=video_id
      )

  response = video_query.execute()

  if not response['items']:
    print("A query came up empty")
    return None

  snippet = response['items'][0]['snippet']
  contentDetails = response['items'][0]['contentDetails']

  return {
      "Title": snippet['title'],
      "Published": snippet['publishedAt'],
      "Channel": snippet['channelTitle'],
      "Description": snippet['description'],
      #"Tags": snippet['tags'] if 'tags' in snippet else 'No Tags',
      "Tags": snippet.get('tags','No Tags'),#Much better way of getting values that can be missing.
      "Captions": contentDetails['caption'],
      "Duration": contentDetails['duration'],
      "Default Language": snippet['defaultLanguage'] if 'defaultLanguage' in snippet else 'No Default Language'
      #Like above, it's better to use .get() leaving this here as a note for myself
  }

In [165]:
meta_data_dicts = []

for channel in recent_trump:
  for video in recent_trump[channel]:
    meta_data_dicts.append(get_video_meta_data(video))

In [171]:
meta_df = pd.DataFrame(meta_data_dicts)
meta_df.Duration = pd.to_timedelta(meta_df.Duration)
meta_df.to_csv('meta_df.csv')
meta_df.head()

,Title,Published,Channel,Description,Tags,Captions,Duration,Default Language
0,"Sunday Scrum | Trump’s new tariff threat, anot...",2026-02-22T18:36:54Z,CBC News,Chief political correspondent Rosemary Barton ...,"[CBC News, CBC, News, breaking news, Canada, S...",false,0 days 00:12:03,en
1,Trump now says he's raising global tariff rate...,2026-02-21T22:08:22Z,CBC News,After imposing a global tariff of 10 per cent ...,"[CBC News, CBC, News, breaking news]",false,0 days 00:00:46,en
2,Trump says he's raising new global tariff rate...,2026-02-21T21:20:51Z,CBC News,U.S. President Donald Trump said on social med...,"[CBC News, CBC, News, breaking news, Trump, ta...",false,0 days 00:02:25,en
3,Transport Canada certifies Gulfstream jets ami...,2026-02-21T17:42:56Z,CBC News,Transport Canada has certified the American-ma...,"[CBC News, CBC, News, breaking news, Trump, ta...",false,0 days 00:02:06,en
4,What do Trump’s new 10% tariffs mean for Canada?,2026-02-21T02:38:53Z,CBC News,U.S. President Donald Trump imposed a new 10 p...,"[CBC News, CBC, News, breaking news]",false,0 days 00:01:30,en


Part 3 - Statistical Information

For each video collected in Part 2, retrieve:
* View Count
* Like Count
* Favorite Count
* Comment Count

In [167]:
def get_video_stats(video_id):
  video_query = youtube.videos().list(
      part='statistics',
      id=video_id
      )

  response = video_query.execute()
  stats = response['items'][0]['statistics']

  return {
      "View Count": stats.get('viewCount', '0'),
      "Like Count": stats.get('likeCount', '0'),
      "Favorite Count": stats.get('favoriteCount', '0'),
      "Comment Count": stats.get('commentCount', '0')
  }

In [168]:
stats_dicts = []

for channel in recent_trump:
  for video in recent_trump[channel]:
    stats_dicts.append(get_video_stats(video))

In [170]:
stats_df = pd.DataFrame(stats_dicts)
stats_df.to_csv('stats_df.csv')
stats_df.head()

,View Count,Like Count,Favorite Count,Comment Count
0,59437,691,0,0
1,28176,233,0,0
2,96853,466,0,0
3,18252,194,0,0
4,137223,1558,0,0


*Worthwhile to note that the favorite count is always 0, this is not an error. It's because favorite count is a deprecated feature, meaning it is no longer tracked. I imagine it is still a property/part of the JSON response so that older code/infrastructure that would rely on it does not break when they made the change, giving time for programmers to adjust their code bases. Comment Count on the other hand appears as 0 when either there are 0 comments or comments were disabled.*

Part 4 - Channel Information

For each of the 10 selected Ccannels retrieve:

* Subscriber Count
* Total Views
* Total Videos
* Country
* Channel Description

In [180]:
def get_channel_data(channel_id):
  channel_query = youtube.channels().list(
      part='statistics,snippet',
      id=channel_id)

  response = channel_query.execute()

  if not response['items']:
    print(f"No data found for channel ID: {channel_id}")
    return None

  stats = response['items'][0]['statistics']
  snippet = response['items'][0]['snippet']

  return {
      "Channel Name" : snippet.get('title', 'No Title'),
      "Subscriber Count": stats.get('subscriberCount', '0'),
      "Total Views": stats.get('viewCount', '0'),
      "Total Videos": stats.get('videoCount', '0'),
      "Country": snippet.get('country', 'No Country'),
      "Channel Description": snippet.get('description', 'No Description')
  }

In [181]:
channel_stats_dicts = []

for channel in channel_ids:
  channel_stats_dicts.append(get_channel_data(channel_ids[channel]))

In [182]:
channel_stats_df = pd.DataFrame(channel_stats_dicts)
channel_stats_df.to_csv('channel_stats_df.csv')
channel_stats_df.head()

,Channel Name,Subscriber Count,Total Views,Total Videos,Country,Channel Description
0,CBC News,4540000,2874258475,43631,CA,As Canada's national public news and informati...
1,USA TODAY,7460000,5068921277,31441,US,From heartwarming moments to the latest in spo...
2,Washington Post,2830000,1633222229,24357,US,"Live coverage, breaking news, analysis and mor..."
3,NBC News,11800000,9651763114,90839,No Country,"Every day, NBC News helps people understand wh..."
4,Reuters,4050000,2218213172,78048,US,"Reuters brings you the latest breaking news, b..."
